# RoboSkills video baseline — Google Colab GPU stages

**Reusable notebook template; execution evidence is recorded separately.** This notebook runs the pinned upstream SAM 3 masks, MoGe-3 depth, and GeoCalib gravity stages on the included sugar-box video. This template requires A100 and defaults to all 70 depth frames with a 900-second stage limit. Set `DEPTH_FRAMES = [0]` for a one-frame smoke test. The initial T4 full-video attempt was interrupted for excessive runtime; an L4 one-frame test subsequently passed. It downloads models and may use several GB of storage. Select **Runtime → Change runtime type → GPU**; availability and limits vary.

**This notebook covers the upstream video perception baseline.** The separate `chess_reconstruction_colab.ipynb` and `chess-colab-input.zip` add the native SAM 3D adapter and the chess simulation/graph/Scenic pipeline. The full hand-motion workflow, including HaWoR and MANO, is separate.

Sources: [RoboSkills pinned source](https://github.com/openrobokit/roboskills/tree/eca4691d2dfbb808dd58f3dd279fed154de6828a), [SAM 3D setup](https://github.com/facebookresearch/sam-3d-objects/blob/main/doc/setup.md), [Colab FAQ](https://research.google.com/colaboratory/faq.html).


In [ ]:
import os, sys, json, shutil, subprocess, time
from pathlib import Path
import torch
if not torch.cuda.is_available():
    raise RuntimeError("Select a Colab GPU runtime before installing or running the GPU models.")
print({"gpu": torch.cuda.get_device_name(0), "memory_GiB": round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1)})
REQUIRED_GPU = "A100"  # User's requested hardware; fail if Colab silently substitutes another GPU.
if REQUIRED_GPU not in torch.cuda.get_device_name(0):
    raise RuntimeError(f"Requested {REQUIRED_GPU}, but Colab assigned {torch.cuda.get_device_name(0)}. Request A100 again; no models were launched.")
ROOT = Path("/content/roboskills")
COMMIT = "eca4691d2dfbb808dd58f3dd279fed154de6828a"
if not ROOT.exists():
    subprocess.run(["git", "clone", "https://github.com/openrobokit/roboskills.git", str(ROOT)], check=True)
subprocess.run(["git", "-C", str(ROOT), "checkout", "--detach", COMMIT], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "uv", "opencv-python-headless", "huggingface_hub", "-e", str(ROOT)], check=True)
sys.path.insert(0, str(ROOT / "src"))
# Upstream model workers each create their own uv environment; keep them sequential.
os.environ["UV_CACHE_DIR"] = "/content/uv-cache"
os.environ["HF_HOME"] = "/content/hf-cache"
os.environ["MUJOCO_GL"] = "egl"
RUN = Path("/content") / ("roboskills-baseline-" + time.strftime("%Y%m%d-%H%M%S"))
RUN.mkdir(exist_ok=False)
VIDEO = ROOT / "assets/example_data/sugar-box.mp4"
assert VIDEO.is_file()
statuses = {}
DEPTH_FRAMES = None
STAGE_TIMEOUT_SECONDS = 900


## SAM 3 access

Request access to `facebook/sam3` on Hugging Face first. In Colab's **Secrets** panel add `HF_TOKEN` and grant this notebook access. Do not paste credentials into code or output. If the secret is absent, SAM 3 is skipped and the non-gated depth/gravity stages can still run. A token's presence does not guarantee model access.


In [ ]:
from google.colab import userdata
try:
    token = userdata.get("HF_TOKEN")
except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
    token = None
if token:
    os.environ["HF_TOKEN"] = token
    del token
print("SAM 3 credential available:", bool(os.environ.get("HF_TOKEN")))


In [ ]:
def run_stage(name, args):
    import signal
    started = time.monotonic()
    log_path = RUN / (name + ".log")
    print("Starting", name, "with timeout", STAGE_TIMEOUT_SECONDS, "seconds", flush=True)
    statuses[name] = {"status": "running", "log": log_path.name}
    (RUN / "status.json").write_text(json.dumps(statuses, indent=2))
    timed_out = False
    with log_path.open("w") as log:
        process = subprocess.Popen([sys.executable, "-m", "roboskills.tools." + name, *map(str, args)],
                                   stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
        try:
            returncode = process.wait(timeout=STAGE_TIMEOUT_SECONDS)
        except subprocess.TimeoutExpired:
            timed_out = True
            os.killpg(process.pid, signal.SIGKILL)
            returncode = process.wait()
        except BaseException:
            os.killpg(process.pid, signal.SIGKILL)
            process.wait()
            raise
    status = {"returncode": returncode, "timed_out": timed_out, "seconds": round(time.monotonic()-started, 1), "log": log_path.name}
    statuses[name] = status
    (RUN / "status.json").write_text(json.dumps(statuses, indent=2))
    print(name, status)
    if returncode:
        # The token is never printed; redact if an upstream error happens to include it.
        tail = log_path.read_text(errors="replace")[-4000:]
        secret = os.environ.get("HF_TOKEN")
        print(tail.replace(secret, "[REDACTED]") if secret else tail)
    return returncode == 0

if os.environ.get("HF_TOKEN"):
    run_stage("sam3", ["--video", VIDEO, "--prompts", "sugar box", "right hand", "--out", RUN / "masks"])
else:
    statuses["sam3"] = {"status": "blocked", "reason": "No approved Hugging Face credential supplied"}
depth_args = ["--video", VIDEO, "--out", RUN / "depth"]
if DEPTH_FRAMES is not None:
    depth_args += ["--frames", *DEPTH_FRAMES]
run_stage("moge3", depth_args)
run_stage("geocalib", ["--video", VIDEO, "--out", RUN / "gravity"])


In [ ]:
import numpy as np
import cv2
sys.path.insert(0, str(ROOT / "src"))
checks = {}
for name, status in ((k, v) for k, v in statuses.items() if k in ("sam3", "moge3", "geocalib")):
    checks[name] = {"passed": False, "reason": status.get("reason", "process failed")}
    if status.get("returncode") != 0:
        continue
    try:
        if name == "sam3":
            from roboskills.tools.sam3.sam3 import check_masks
            check = check_masks(RUN / "masks", 0, ["sugar box", "right hand"])
            checks[name] = {"passed": bool(check["ok"]), **check}
        elif name == "moge3":
            cap = cv2.VideoCapture(str(VIDEO))
            n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            cap.release()
            with np.load(RUN / "depth/depth.npz", allow_pickle=False) as data:
                expected = np.arange(n) if DEPTH_FRAMES is None else np.array(sorted({0, *DEPTH_FRAMES}))
                assert len(data["depth"]) == len(expected) and np.array_equal(data["frames"], expected)
                assert data["K"].shape == (3, 3) and np.isfinite(data["K"]).all()
                assert np.isfinite(data["depth"]).all() and np.all(data["depth"][data["valid"]] > 0)
                checks[name] = {"passed": True, "frames": len(expected), "source_frames": n,
                                "frame_indices": expected.tolist(), "full_video": len(expected) == n,
                                "valid_fraction": float(data["valid"].mean()),
                                "metric_scale": "model estimate; not measured LiDAR validation"}
        elif name == "geocalib":
            gravity = json.loads((RUN / "gravity/gravity.json").read_text())
            down = np.asarray(gravity["down"], float)
            assert down.shape == (3,) and np.isfinite(down).all() and abs(np.linalg.norm(down)-1) < 0.05
            checks[name] = {"passed": True, "down": down.tolist()}
    except Exception as error:
        checks[name] = {"passed": False, "reason": str(error)}
report = {"upstream_commit": COMMIT, "checks": checks, "full_reconstruction": False,
          "gpu": torch.cuda.get_device_name(0), "stage_statuses": statuses,
          "pending": ["SAM 3D", "HaWoR", "alignment", "tracking", "collision decomposition", "scene assembly"]}
(RUN / "verification.json").write_text(json.dumps(report, indent=2))
print(json.dumps(report, indent=2))


## Preview the depth result

Estimated video depth is separate from the measured iPhone-depth path. Do not substitute it silently for measured LiDAR or treat the estimated scale as validated.


In [ ]:
from IPython.display import display
from PIL import Image
if checks.get("moge3", {}).get("passed"):
    cap = cv2.VideoCapture(str(VIDEO))
    ok, bgr = cap.read()
    cap.release()
    assert ok
    with np.load(RUN / "depth/depth.npz", allow_pickle=False) as d:
        z = d["depth"][0].astype(float)
        valid = d["valid"][0]
        lo, hi = np.percentile(z[valid], [2, 98])
        normalized = np.clip((z-lo)/max(hi-lo, 1e-6)*255, 0, 255).astype(np.uint8)
        color = cv2.applyColorMap(normalized, cv2.COLORMAP_TURBO)[..., ::-1]
        color[~valid] = 0
    preview = Image.fromarray(np.concatenate([bgr[..., ::-1], color], axis=1))
    preview.save(RUN / "depth_preview.png")
    display(preview)


## Original hand-motion workflow prerequisites (not the chess adapter)

On a supported Linux NVIDIA host, provision the upstream weights with `bash docker/weights.sh sam3d` and `bash docker/weights.sh hawor`, after model access is granted. The full launcher command is printed below. Docker CLI presence alone does not prove a working daemon or GPU passthrough. The original launcher still requires its container runtime. The separate chess notebook runs the unchanged SAM 3D worker using the published image's extracted Python environment.

The upstream SAM 3D offload wrapper reports about 8.9 GB GPU use; the original model's default installation asks for more. This is an upstream measurement, not a guarantee for a Colab runtime. GPU assignment and capacity vary.


In [ ]:
weights = Path.home() / ".roboskills/weights"
full_prerequisites = {"docker_cli": shutil.which("docker") is not None,
    "sam3d_weights": (weights / "sam3d/checkpoints/pipeline.yaml").is_file(),
    "hawor_weights": (weights / "hawor/hawor/checkpoints/hawor.ckpt").is_file()}
print(full_prerequisites)
print("Full reconstruction remains unexecuted. Run this only on a validated Linux NVIDIA Docker host:")
import shlex
command = [sys.executable, "-m", "roboskills.workflows.real2sim", "--workspace", str(RUN / "full"),
    "--video", str(VIDEO), "--object", "sugar box", "--tools", "sam3", "moge3", "geocalib", "sam3d", "hawor",
    "align", "fit", "track", "coacd", "mujoco", "--support", "pedestal", "--retry", "--no-parallel"]
print(shlex.join(command))


## Download your evidence

The archive contains generated results and logs, not credentials or model caches. It is the handoff for reviewing which stages passed. Colab runtimes are temporary; download it before disconnecting.


In [ ]:
from google.colab import files
# Redact secrets from logs before packaging, even if a dependency unexpectedly logged one.
secret = os.environ.get("HF_TOKEN")
if secret:
    for log in RUN.glob("*.log"):
        log.write_text(log.read_text(errors="replace").replace(secret, "[REDACTED]"))
archive = shutil.make_archive(str(RUN), "zip", RUN)
files.download(archive)
